[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/clase/S3_clasificacion_arboles.ipynb) · [Ver en el sitio](https://hesusg.github.io/dammg-supervisado/notebooks/clase/S3_clasificacion_arboles.html) · [Menú de notebooks](https://hesusg.github.io/dammg-supervisado/notebooks/)

# Sesión 3 · Clasificación y árboles, en chiquito
**DAMMG · Módulo VI · vie 6 nov** · 5 mini-labs de unos 15–20 min cada uno · **no se entrega ni se califica**

| # | Mini-lab | Idea clave |
|---|---|---|
| 1 | Repaso: regresión logística | Una recta parte el plano y da una probabilidad |
| 2 | KNN (nuevo) | Dime quiénes son tus vecinos y te diré quién eres |
| 3 | Matriz de confusión y umbral | El umbral lo eliges tú, según qué error te duele más |
| 4 | Un árbol es una serie de preguntas | La mejor pregunta es la que deja los grupos más "puros" |
| 5 | El árbol que memoriza, y el bosque | Un árbol solo sobreajusta; muchos árboles votando, menos |

> **Cómo usar este notebook**
> 1. *Archivo → Guardar una copia en Drive* y trabaja en tu copia.
> 2. Ejecuta las celdas **en orden** con `Shift + Enter`.
> 3. Solo cambias código en las celdas marcadas **✋ Tu turno**.
> 4. Las preguntas **💬 Para el chat** se responden en el chat de Zoom, con una frase basta.
>
> Si algo falla: *Entorno de ejecución → Reiniciar y ejecutar todo*.

In [ ]:
# Preparación: ejecuta esta celda una vez. No necesitas leerla.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

sns.set_theme(style="whitegrid", context="notebook")
AZUL, ROJO = "#4C72B0", "#C44E52"


def dibuja_frontera(modelo, X, y, ax=None, titulo="", nombres=("clase 0", "clase 1")):
    """Pinta de color la zona que el modelo asigna a cada clase y encima los puntos."""
    ax = ax or plt.gca()
    c0, c1 = X.columns[:2]
    xx, yy = np.meshgrid(
        np.linspace(X[c0].min() - 0.5, X[c0].max() + 0.5, 250),
        np.linspace(X[c1].min() - 0.5, X[c1].max() + 0.5, 250),
    )
    malla = pd.DataFrame({c0: xx.ravel(), c1: yy.ravel()})
    zona = modelo.predict(malla).reshape(xx.shape)
    ax.contourf(xx, yy, zona, levels=[-0.5, 0.5, 1.5], cmap=ListedColormap([AZUL, ROJO]), alpha=0.2)
    etiquetas = pd.Series(np.asarray(y)).map({0: nombres[0], 1: nombres[1]})
    sns.scatterplot(x=X[c0].values, y=X[c1].values, hue=etiquetas.values, hue_order=list(nombres),
                    palette={nombres[0]: AZUL, nombres[1]: ROJO}, s=60, edgecolor="k", ax=ax)
    ax.set(xlabel=c0, ylabel=c1, title=titulo)
    return ax

print("✅ entorno listo")

---
## 1 · Repaso: regresión logística

🎯 **Idea clave.** La logística traza una **recta** que separa dos clases y, para cada punto, da una **probabilidad** entre 0 y 1. Más lejos de la recta, más segura está.

Trabajaremos con 20 estudiantes inventados. De cada quien sabemos cuántas horas estudió, cuántas durmió y si **reprobó** (1) o no (0). La meta: detectar a tiempo a quien va a reprobar para ofrecerle tutoría.

In [ ]:
alumnos = pd.DataFrame({
    "estudio": [1, 2, 2, 3, 3, 4, 4, 5, 5, 5, 6, 6, 7, 7, 8, 8, 9, 3, 6, 1],
    "sueno":   [4, 5, 8, 4, 6, 5, 8, 3, 6, 8, 4, 7, 5, 8, 4, 6, 7, 8, 5, 7],
    "reprueba":[1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1],
})
X = alumnos[["estudio", "sueno"]]
y = alumnos["reprueba"]
alumnos.head()

Primero, mira los datos. Cada punto es un estudiante; rojo es quien reprobó.

In [ ]:
sns.scatterplot(data=alumnos, x="estudio", y="sueno", hue="reprueba",
                palette={0: AZUL, 1: ROJO}, s=80, edgecolor="k")
plt.title("¿Quién reprueba?");

Entrenamos la logística con una línea y vemos la frontera que encontró.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistica = LogisticRegression().fit(X, y)
dibuja_frontera(logistica, X, y, titulo="Logística: la frontera es una recta",
                nombres=("aprueba", "reprueba"));

La logística no solo dice "sí" o "no": da una probabilidad. Preguntemos por una estudiante nueva.

✋ **Tu turno.** Cambia las horas de la estudiante nueva y vuelve a ejecutar. ¿Con cuántas horas de estudio baja de 50 %?

In [ ]:
nueva = pd.DataFrame({"estudio": [4], "sueno": [6]})   # ✋ CAMBIA ESTOS NÚMEROS

prob = logistica.predict_proba(nueva)[0, 1]
print(f"Probabilidad de reprobar: {prob:.0%}")

💬 **Para el chat.** Completa: *"Si una estudiante duerme 6 horas, necesita estudiar al menos ___ horas para que el modelo le dé menos de 50 %."*

---
## 2 · KNN: k vecinos más cercanos (nuevo)

🎯 **Idea clave.** Para clasificar a alguien nuevo, KNN busca a los **k** estudiantes más parecidos (los más cercanos en la gráfica) y deja que **voten**. No aprende ninguna recta: solo guarda los datos y mide distancias.

Su apuesta (su *sesgo inductivo*): **lo que está cerca se parece**.

### 2a · Hazlo a mano
La distancia es la de la regla: $\sqrt{(\Delta\,\text{estudio})^2 + (\Delta\,\text{sueño})^2}$. Calculémosla del estudiante nuevo a los 20.

In [ ]:
nuevo = {"estudio": 4, "sueno": 6}

alumnos["distancia"] = np.sqrt((alumnos["estudio"] - nuevo["estudio"])**2 +
                               (alumnos["sueno"] - nuevo["sueno"])**2)
vecinos = alumnos.sort_values("distancia").head(3)   # los 3 más cercanos
vecinos

In [ ]:
votos = vecinos["reprueba"].sum()
print(f"De 3 vecinos, {votos} reprobaron → KNN dice:", "reprueba" if votos >= 2 else "aprueba")

### 2b · Lo mismo con scikit-learn
`KNeighborsClassifier` hace exactamente esa cuenta. Comprobemos que da lo mismo.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=3).fit(X, y)
print("scikit-learn dice:", knn.predict(pd.DataFrame([nuevo]))[0], "(1 = reprueba)")

### 2c · ¿Qué hace k?
Con **k = 1** cada punto manda sobre su zonita: la frontera se vuelve un rompecabezas que memoriza hasta el ruido. Con **k grande** votan muchos y la frontera se suaviza.

✋ **Tu turno.** Cambia los dos valores de `k` (prueba 1, 3, 7, 15 y 19) y compara.

In [ ]:
k_chico, k_grande = 1, 15        # ✋ CAMBIA ESTOS NÚMEROS

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for ax, k in zip(ejes, [k_chico, k_grande]):
    modelo = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    dibuja_frontera(modelo, X, y, ax=ax, titulo=f"KNN con k = {k}", nombres=("aprueba", "reprueba"))
plt.tight_layout()

> **Ojo con las escalas.** KNN mide distancias, así que si una variable va de 0 a 10 y otra de 0 a 10,000, la segunda manda sola. Con datos reales siempre se **escala** primero (lo vieron en el Módulo IV). Aquí no hace falta: las dos columnas son horas.

💬 **Para el chat.** ¿Qué k elegirías y por qué? Una frase.

---
## 3 · Matriz de confusión y umbral

🎯 **Idea clave.** El modelo da una probabilidad; **tú** decides a partir de qué número mandas a tutoría. Ese número es el **umbral**. Si lo bajas, alcanzas a más de quienes reprueban, pero también mandas a tutoría a quien no la necesitaba.

> Para ver la mecánica usamos los mismos 20 estudiantes con los que entrenamos. En un caso real esto se mide en **validación** (como en el Lab 2).

### 3a · Del número a la decisión

In [ ]:
alumnos["prob"] = logistica.predict_proba(X)[:, 1]

umbral = 0.5                     # ✋ más adelante cambiarás este número
alumnos["alerta"] = (alumnos["prob"] >= umbral).astype(int)
alumnos[["estudio", "sueno", "reprueba", "prob", "alerta"]].round(2)

### 3b · La matriz de confusión: contar aciertos y errores
Hay cuatro casillas: alerté y sí reprobó (**VP**), alerté y no reprobó (**FP**), no alerté y sí reprobó (**FN**), no alerté y no reprobó (**VN**).

In [ ]:
matriz = pd.crosstab(alumnos["reprueba"], alumnos["alerta"],
                     rownames=["¿reprobó?"], colnames=["¿alerta?"])
sns.heatmap(matriz, annot=True, cmap="Blues", cbar=False, annot_kws={"size": 16})
plt.title(f"Matriz de confusión (umbral = {umbral})");

### 3c · Dos preguntas, dos números
- **Precision:** de los que mandé a tutoría, ¿cuántos sí la necesitaban? → $\frac{VP}{VP + FP}$
- **Recall:** de los que reprobaron, ¿a cuántos alcancé? → $\frac{VP}{VP + FN}$

In [ ]:
VP = ((alumnos["reprueba"] == 1) & (alumnos["alerta"] == 1)).sum()
FP = ((alumnos["reprueba"] == 0) & (alumnos["alerta"] == 1)).sum()
FN = ((alumnos["reprueba"] == 1) & (alumnos["alerta"] == 0)).sum()

print(f"VP = {VP}, FP = {FP}, FN = {FN}")
print(f"Precision = {VP / (VP + FP):.0%}   Recall = {VP / (VP + FN):.0%}")

### 3d · Mueve el umbral
La celda prueba varios umbrales a la vez para que veas el intercambio.

✋ **Tu turno.** Agrega o quita umbrales de la lista. ¿Cuál elegirías si tu escuela **solo puede dar 8 tutorías**?

In [ ]:
umbrales = [0.2, 0.35, 0.5, 0.65, 0.8]     # ✋ CAMBIA LA LISTA

filas = []
for u in umbrales:
    alerta = alumnos["prob"] >= u
    vp = (alerta & (alumnos["reprueba"] == 1)).sum()
    filas.append({"umbral": u,
                  "alertas": alerta.sum(),
                  "precision": vp / max(alerta.sum(), 1),
                  "recall": vp / (alumnos["reprueba"] == 1).sum()})
pd.DataFrame(filas).round(2)

### 3e · Todos los umbrales en una sola curva: ROC
Si pruebas **todos** los umbrales y para cada uno anotas cuántos de los que reprobaron alcanzas (eje y) contra cuántos de los que no reprobaron alertas por error (eje x), sale la **curva ROC**. La diagonal es lanzar una moneda. El área bajo la curva (**AUC**) resume todo en un número: 1 es perfecto y 0.5 es la moneda.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, _ = roc_curve(alumnos["reprueba"], alumnos["prob"])
sns.lineplot(x=fpr, y=tpr, drawstyle="steps-post", linewidth=2.5, label="logística")
plt.plot([0, 1], [0, 1], "--", color="gray", label="moneda")
plt.xlabel("Falsas alarmas (de los que NO reprobaron)")
plt.ylabel("Alcanzados (de los que SÍ reprobaron)")
plt.title(f"Curva ROC · AUC = {roc_auc_score(alumnos['reprueba'], alumnos['prob']):.2f}");

💬 **Para el chat.** En este caso, ¿qué error te duele más: mandar a tutoría a alguien que no la necesitaba (FP) o no alcanzar a alguien que iba a reprobar (FN)? ¿Subes o bajas el umbral?

---
## 4 · Un árbol es una serie de preguntas

🎯 **Idea clave.** Un árbol de decisión clasifica haciendo preguntas de sí o no ("¿llueve?"). Para escoger **qué preguntar primero**, mide qué tan mezclados quedan los grupos. La mejor pregunta es la que deja los grupos más **puros**.

Diez días, y si se jugó fútbol o no:

In [ ]:
dias = pd.DataFrame({
    "llueve": [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
    "viento": [0, 0, 1, 1, 0, 1, 0, 1, 0, 1],
    "juega":  [1, 1, 1, 1, 1, 0, 0, 0, 1, 0],
})
dias

### 4a · Medir la mezcla: el índice de Gini
Con una proporción $p$ de "sí", el Gini es $1 - p^2 - (1-p)^2$.
- Grupo puro (todos sí o todos no): **Gini = 0**.
- Mitad y mitad: **Gini = 0.5**, la mezcla máxima.

✋ **Tu turno.** Prueba la función con otras proporciones.

In [ ]:
def gini(p):
    return 1 - p**2 - (1 - p)**2

print("todos sí  :", gini(1.0))
print("mitad     :", gini(0.5))
print("7 de 10 sí:", round(gini(0.7), 2))     # ✋ PRUEBA OTRAS PROPORCIONES

### 4b · ¿Qué pregunta conviene más?
Para cada pregunta partimos los días en dos grupos, medimos el Gini de cada grupo y sacamos un **promedio ponderado** por tamaño. Gana la pregunta con el número más bajo.

In [ ]:
print("Gini antes de preguntar:", round(gini(dias["juega"].mean()), 3))

for pregunta in ["llueve", "viento"]:
    total = 0
    for valor, grupo in dias.groupby(pregunta):
        peso = len(grupo) / len(dias)
        total += peso * gini(grupo["juega"].mean())
    print(f"Si pregunto '¿{pregunta}?' → Gini promedio = {total:.3f}")

### 4c · ¿Qué elige scikit-learn?
Un árbol de profundidad 1 (una sola pregunta) debería escoger la misma que tú.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

arbol = DecisionTreeClassifier(max_depth=1).fit(dias[["llueve", "viento"]], dias["juega"])
plt.figure(figsize=(8, 5))
plot_tree(arbol, feature_names=["llueve", "viento"], class_names=["no juega", "juega"],
          filled=True, impurity=True);

💬 **Para el chat.** ¿Qué pregunta eligió el árbol y con qué Gini? ¿Coincide con tu cuenta?

---
## 5 · El árbol que memoriza, y el bosque que lo arregla

🎯 **Idea clave.** Si dejas que un árbol pregunte y pregunte, termina con una hoja para cada punto: **memoriza** los datos de entrenamiento, ruido incluido, y falla con datos nuevos. Un **bosque aleatorio** entrena muchos árboles, cada uno con una muestra distinta, y los pone a votar. Los errores de cada uno se cancelan.

Usamos dos "lunas" con ruido: 200 puntos que ninguna recta separa bien. Guardamos 30 % aparte como **prueba**, para medir cómo le va con datos que no vio.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

Xl, yl = make_moons(n_samples=200, noise=0.3, random_state=0)
Xl = pd.DataFrame(Xl, columns=["x1", "x2"])
X_ent, X_prueba, y_ent, y_prueba = train_test_split(Xl, yl, test_size=0.3, random_state=0)

sns.scatterplot(x=Xl["x1"], y=Xl["x2"], hue=yl, palette={0: AZUL, 1: ROJO}, s=50, edgecolor="k")
plt.title("Dos lunas con ruido");

### 5a · Más profundidad: ¿mejor o peor?
Entrenamos árboles de profundidad 1 a 12 y medimos la **exactitud** (porcentaje de aciertos) en entrenamiento y en prueba.

In [ ]:
filas = []
for prof in range(1, 13):
    a = DecisionTreeClassifier(max_depth=prof, random_state=0).fit(X_ent, y_ent)
    filas.append({"profundidad": prof, "conjunto": "entrenamiento", "exactitud": a.score(X_ent, y_ent)})
    filas.append({"profundidad": prof, "conjunto": "prueba", "exactitud": a.score(X_prueba, y_prueba)})

sns.lineplot(data=pd.DataFrame(filas), x="profundidad", y="exactitud", hue="conjunto", marker="o")
plt.title("Entrenamiento sube siempre; prueba no");

### 5b · Árbol contra bosque
✋ **Tu turno.** Cambia la profundidad del árbol (`prof`) y el número de árboles del bosque (`n_arboles`). ¿Cuál frontera se ve más "razonable"?

In [ ]:
from sklearn.ensemble import RandomForestClassifier

prof, n_arboles = 3, 200          # ✋ CAMBIA ESTOS NÚMEROS

modelos = {
    f"Árbol, profundidad {prof}": DecisionTreeClassifier(max_depth=prof, random_state=0),
    "Árbol sin límite": DecisionTreeClassifier(random_state=0),
    f"Bosque de {n_arboles} árboles": RandomForestClassifier(n_estimators=n_arboles, random_state=0),
}
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (nombre, m) in zip(ejes, modelos.items()):
    m.fit(X_ent, y_ent)
    dibuja_frontera(m, X_ent, y_ent, ax=ax, titulo=f"{nombre}\nprueba: {m.score(X_prueba, y_prueba):.0%}")
plt.tight_layout()

### 5c · La tabla final, con boosting
El **boosting** también usa muchos árboles, pero en fila: cada árbol nuevo corrige los errores del anterior.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

modelos["Boosting"] = GradientBoostingClassifier(random_state=0)
tabla = []
for nombre, m in modelos.items():
    m.fit(X_ent, y_ent)
    tabla.append({"modelo": nombre,
                  "entrenamiento": m.score(X_ent, y_ent),
                  "prueba": m.score(X_prueba, y_prueba)})
pd.DataFrame(tabla).round(2)

💬 **Para el chat.** El "árbol sin límite" saca casi 100 % en entrenamiento. ¿Por qué no es el mejor modelo?

---
### Para llevarte
1. **KNN** decide por votación de vecinos; **k chico memoriza, k grande suaviza**.
2. El **umbral** es una decisión tuya: precision y recall se intercambian, y la ROC resume todos los umbrales.
3. Un **árbol** elige la pregunta que deja los grupos más puros (menor Gini). Solo, sobreajusta; en **bosque**, mejora.

Lo que sigue: **Lab 3** (se entrega el dom 15 nov) hace esto mismo con los datos reales de MiBici.